# Исследование восьми алгоритмов генерации случайных величин

Петров Мирон и Каплунова Анна · Fifty Shades of Coffee, S2

1. [Методы из проекта](#методы-из-проекта): обратное преобразование, псевдонимы, отбор с отклонением, суперпозиция.
2. [Специализированные методы](#специализированные-методы): Бокс–Мюллер, Марсалья–Цанг для гамма-распределения, алгоритм Кнута для распределения Пуассона и Ziggurat.
3. [Результаты контрольных запусков](#результаты-контрольных-запусков).

Во всех примерах NumPy даёт исходные равномерные числа. Мы исследуем, как из них получать заданные распределения. Смена исходного генератора или его `seed` сама по себе не даёт нового метода преобразования. Некоторые из восьми алгоритмов родственны: отбор с отклонением - общая схема, а методы Марсальи–Цанга и Ziggurat используют свои правила предложения и принятия.

Первые четыре метода связаны с [основным S2](stat-lab1-s2.ipynb) и [ноутбуком с дополнительными методами](stat-lab1-s2-bonus-alias.ipynb). Здесь для демонстрации группируем влажность в три диапазона. В проекте генераторы сохраняют исходные уровни влажности. Последние четыре метода демонстрируем на контрольных задачах с известными законами. Эти примеры не означают, что влажность кофе описывается указанными законами.

Для запуска выполняем все ячейки по порядку в Jupyter или Colab с NumPy. Связанные данные и результаты проекта - в [сводке S2](S2-RESULTS.md).

In [ ]:
# Входные частоты и параметры эксперимента
import math

import numpy as np

N = 20_000
levels = np.array([0, 1, 2])
labels = ["Ниже 10%", "10–12%", "Выше 12%"]
all_counts = np.array([92, 844, 122])
mexico_counts = np.array([3, 148, 71])
guatemala_counts = np.array([9, 162, 3])
p_all = all_counts / all_counts.sum()
p_mx = mexico_counts / mexico_counts.sum()
p_gt = guatemala_counts / guatemala_counts.sum()

for name, counts in [("Вся арабика", all_counts), ("Мексика", mexico_counts),
                     ("Гватемала", guatemala_counts)]:
    print(name, ":", dict(zip(labels, counts.tolist())))

Вся арабика : {'Ниже 10%': 92, '10–12%': 844, 'Выше 12%': 122}
Мексика : {'Ниже 10%': 3, '10–12%': 148, 'Выше 12%': 71}
Гватемала : {'Ниже 10%': 9, '10–12%': 162, 'Выше 12%': 3}


## Методы из проекта

### 1. Обратное преобразование

**Распределения и принцип.** Для непрерывного закона преобразуем равномерное число $U$ через обратную функцию распределения. Для дискретного закона делим интервал от 0 до 1 на участки длины $p_i$. Например, в нашей таблице интервалу 10–12% достаётся $844/1058$ всей длины, поэтому такой же будет вероятность получить эту категорию.

**Где используется.** В основном S2 по накопленным частотам выбираем уровень влажности. С непрерывным законом по той же схеме можно, например, генерировать экспоненциальное время ожидания, если известна его формула.

**Сильная сторона.** Достаточно одного равномерного числа на результат; в дискретном случае поиск по заранее накопленным частотам занимает $O(\log k)$ для $k$ категорий. **Слабая сторона.** Если непрерывную функцию трудно обратить, приходится искать значение численно. Для дискретной таблицы нужно хранить накопленные вероятности.

**Источник:** Л. Девруа, [*Non-Uniform Random Variate Generation*, §1.1 и §1.4](https://luc.devroye.org/handbooksimulation1.pdf).

In [ ]:
# Обратное преобразование
def inversion(rng, cumulative, size):
    return np.searchsorted(cumulative, rng.random(size), side="right")


cdf_all = np.cumsum(p_all)
cdf_all[-1] = 1.0
s1_inverse = inversion(np.random.default_rng(101), cdf_all, N)
print("Первые результаты:", [labels[i] for i in s1_inverse[:8]])

Первые результаты: ['Выше 12%', '10–12%', '10–12%', '10–12%', '10–12%', 'Выше 12%', '10–12%', '10–12%']


### 2. Метод псевдонимов Уокера - Воуза

**Распределения и принцип.** Метод предназначен для конечных дискретных законов. Каждой категории выделяем одну ячейку. Если её вероятности не хватает, чтобы заполнить ячейку, свободное место отдаём другой категории - псевдониму. Случайно выбираем ячейку, затем проверяем, чья часть этой ячейки выпала.

**Где используется.** Многократный выбор с фиксированными весами: например, генерация категорий влажности для набора симуляций.

**Сильная сторона.** После подготовки таблицы один результат получается за $O(1)$. **Слабая сторона.** Подготовка и память стоят $O(k)$. Если веса часто меняются, подготовку приходится повторять. Для трёх категорий преимущество в скорости в этом ноутбуке не измеряем.

**Источники:** М. Воуз, [*A Linear Algorithm for Generating Random Numbers with a Given Distribution*](https://doi.org/10.1109/32.92917), 1991; Л. Девруа, [§1.5](https://luc.devroye.org/handbooksimulation1.pdf).

In [3]:
# @title Псевдонимы: точный перебор и запуск { display-mode: "form" }
def alias_table(counts):
    k, total = len(counts), int(counts.sum())
    mass = counts.astype(np.int64) * k
    cut = np.full(k, total)
    alias = np.arange(k)
    small = [i for i in range(k) if mass[i] < total]
    large = [i for i in range(k) if mass[i] >= total]
    while small and large:
        i, j = small.pop(), large.pop()
        cut[i], alias[i] = mass[i], j
        mass[j] -= total - mass[i]
        (small if mass[j] < total else large).append(j)
    return cut, alias, total


cut, alias, total = alias_table(all_counts)
columns, slots = np.indices((len(all_counts), total))
exact = np.where(slots < cut[columns], columns, alias[columns])
np.testing.assert_array_equal(np.bincount(exact.ravel()), len(all_counts)*all_counts)

rng = np.random.default_rng(102)
column = rng.integers(len(all_counts), size=N)
slot = rng.integers(total, size=N)
s1_alias = np.where(slot < cut[column], column, alias[column])
print("Точный перебор ячеек:", np.bincount(exact.ravel()).tolist())

Точный перебор ячеек: [276, 2532, 366]


### 3. Отбор с отклонением фон Неймана

**Распределения и принцип.** Подходит и для дискретных, и для непрерывных законов. Кандидат приходит из простого распределения $q$. Если $p(x)\le Mq(x)$, принимаем кандидата с вероятностью $p(x)/(Mq(x))$. После принятия значения распределены по целевому закону $p$. Для наших трёх категорий предлагаем каждую одинаково часто и принимаем с вероятностью $c_i/\max_j c_j$.

**Где используется.** Когда вероятность или плотность можно вычислить, а обратная функция неудобна. Пример проекта - генерация влажности по дискретным частотам.

**Сильная сторона.** Достаточно сравнивать отношение вероятностей или плотностей. **Слабая сторона.** Среднее число попыток - $M$. Плохое распределение кандидатов даёт много отказов, а слишком маленькая граница $M$ искажает результат.

**Источник:** Л. Девруа, [§1.7–1.8: общий принцип и примеры](https://luc.devroye.org/handbooksimulation1.pdf).

In [ ]:
# Отбор с отклонением: частоты и среднее число попыток
def rejection(rng, counts, size):
    result = np.empty(size, dtype=int)
    attempts = 0
    for i in range(size):
        while True:
            attempts += 1
            j = rng.integers(len(counts))
            if rng.integers(int(counts.max())) < counts[j]:
                result[i] = j
                break
    return result, attempts


s1_reject, attempts = rejection(np.random.default_rng(103), all_counts, N)
expected_attempts = len(all_counts) * all_counts.max() / all_counts.sum()
print(f"Попыток на значение: теория {expected_attempts:.3f}, запуск {attempts/N:.3f}")

Попыток на значение: теория 2.393, запуск 2.383


### 4. Суперпозиция распределений

**Распределения и принцип.** Подходит для смесей дискретных или непрерывных законов. Сначала выбираем компонент, затем генерируем из него. Вероятность события $A$ равна $\sum_j w_j P(A\mid j)$, где $w_j$ - веса компонентов.

**Где используется.** В нашем S2 смешиваем записи из разных стран. Для примера задаём 80% Мексики и 20% Гватемалы, затем выбираем категорию влажности по частотам выбранной страны. Это смесь распределений записей, а не среднее двух измерений влажности.

**Сильная сторона.** Можно менять доли стран, не подгоняя их профили заново. **Слабая сторона.** Нужны обоснованные веса и генератор для каждого компонента. Ошибки в условных распределениях переходят в смесь.

**Источник:** Л. Девруа, [§1.6: *Mixture methods*](https://luc.devroye.org/handbooksimulation1.pdf).

In [ ]:
# Суперпозиция: 80% Мекс и 20% Гват
rng = np.random.default_rng(104)
is_mx = rng.random(N) < .8
s1_mix = np.empty(N, dtype=int)
for mask, p in [(is_mx, p_mx), (~is_mx, p_gt)]:
    cdf = np.cumsum(p)
    cdf[-1] = 1
    s1_mix[mask] = inversion(rng, cdf, mask.sum())

p_mix = .8*p_mx + .2*p_gt
print(f"Мексика в запуске: {100*is_mx.mean():.2f}%")
print(f"Влажность 10–12%: расчёт {100*p_mix[1]:.2f}%, запуск {100*(s1_mix == 1).mean():.2f}%")

Мексика в запуске: 80.16%
Влажность 10–12%: расчёт 71.95%, запуск 71.45%


## Специализированные методы

### 5. Преобразование Бокса–Мюллера

**Распределения и принцип.** Получаем два независимых значения $N(0,1)$ из независимых $U_1,U_2$. Берём радиус $R=\sqrt{-2\ln U_1}$ и угол $\theta=2\pi U_2$, затем возвращаем $R\cos\theta$ и $R\sin\theta$. Для $N(\mu,\sigma^2)$ умножаем на $\sigma$ и прибавляем $\mu$.

**Где используется.** Например, для имитации независимых ошибок прибора, если предполагаем нормальный шум.

**Сильная сторона.** Два результата за одно преобразование, без повторных попыток. **Слабая сторона.** Нужно вычислять логарифм, корень и тригонометрические функции. Первый равномерный аргумент не должен быть равен нулю.

**Источник:** Дж. Бокс и М. Мюллер, [*A Note on the Generation of Random Normal Deviates*](https://doi.org/10.1214/aoms/1177706645), 1958.

In [ ]:
# Бокс–Мюллер: нормальный шум прибора
def box_muller(rng, size):
    n_pairs = (size + 1) // 2
    u1 = 1 - rng.random(n_pairs)  # (0, 1], логарифм определён
    angle = 2*math.pi*rng.random(n_pairs)
    radius = np.sqrt(-2*np.log(u1))
    return np.column_stack((radius*np.cos(angle), radius*np.sin(angle))).ravel()[:size]


normal = box_muller(np.random.default_rng(105), N)
print("Пример поправки к показанию прибора, п. п.:", np.round(.2*normal[:6], 3))

Пример поправки к показанию прибора, п. п.: [ 0.111  0.263 -0.401  0.469  0.003  0.09 ]


### 6. Метод Марсальи–Цанга для гамма-распределения

**Распределения и принцип.** Для гамма-распределения с параметром формы $\alpha\ge1$ предлагаем кандидата, преобразуя стандартное нормальное значение. Кандидата проверяем по специальному порогу. Принятый результат имеет единичный масштаб; для масштаба $\theta$ умножаем его на $\theta$. В примере используем форму 2 и масштаб 1. Случай $0<\alpha<1$ требует отдельного преобразования и в этом примере не реализован.

**Где используется.** Например, для имитации положительных времён ожидания, если нужна гамма-модель. При форме 2 это время до второго события пуассоновского потока.

**Сильная сторона.** Предложение подобрано для гамма-закона, поэтому вероятность принятия высокая. **Слабая сторона.** Нужен нормальный генератор; число попыток случайно. Формулы зависят от параметра формы.

**Источник:** Дж. Марсалья и В. Цанг, [*A Simple Method for Generating Gamma Variables*](https://doi.org/10.1145/358407.358414), 2000.

In [ ]:

def gamma_mt(rng, shape, size):
    if shape < 1:
        raise ValueError("Этот вариант требует форму не меньше 1")
    d = shape - 1/3
    c = 1/math.sqrt(9*d)
    result = np.empty(size)
    for i in range(size):
        while True:
            z = rng.standard_normal()
            root = 1 + c*z
            if root <= 0:
                continue
            v = root**3
            u = 1 - rng.random()
            if math.log(u) <= z*z/2 + d*(1-v+math.log(v)):
                result[i] = d*v
                break
    return result


gamma = gamma_mt(np.random.default_rng(106), 2, N)
print("Пример времени до второго события:", np.round(gamma[:6], 3))

Пример времени до второго события: [3.086 2.386 3.234 3.592 1.441 0.324]


### 7. Алгоритм Кнута для распределения Пуассона

**Распределения и принцип.** Генерируем число событий за фиксированное время, $X\sim\operatorname{Poisson}(\lambda)$. Последовательно перемножаем независимые равномерные числа, пока произведение не станет меньше или равно $e^{-\lambda}$. Результат - число умножений минус одно. Если взять отрицательные логарифмы, это равносильно подсчёту, сколько экспоненциальных интервалов успело поместиться в отрезок длины $\lambda$.

**Где используется.** Например, для имитации числа попаданий фотонов на детектор за короткий интервал при постоянной интенсивности и независимых событиях.

**Сильная сторона.** Короткий алгоритм без таблиц, удобный при малом $\lambda$. **Слабая сторона.** Среднее число умножений равно $\lambda+1$. При большом $\lambda$ алгоритм замедляется, а экспонента и произведение могут стать слишком малы для точного представления.

**Источник:** [*Reexamining Key Applications of the Poisson Distribution*, Algorithm 1](https://doi.org/10.3390/sym17111828); [Л. Девруа: связь с экспоненциальными интервалами](https://luc.devroye.org/handbooksimulation1.pdf).

In [ ]:
# Кнут: число событий за интервал
def poisson_knuth(rng, rate, size):
    if not 0 <= rate <= 30:
        raise ValueError("В примере используем малую или умеренную интенсивность")
    limit = math.exp(-rate)
    result = np.empty(size, dtype=int)
    for i in range(size):
        product = rng.random()
        count = 0
        while product > limit:
            product *= rng.random()
            count += 1
        result[i] = count
    return result


poisson = poisson_knuth(np.random.default_rng(107), 4, N)
print("Число событий в первых интервалах:", poisson[:10].tolist())

Число событий в первых интервалах: [5, 4, 1, 4, 4, 4, 3, 1, 4, 2]


### 8. Ziggurat для экспоненциального закона

**Распределения и принцип.** Метод используют для стандартного экспоненциального и нормального законов. В примере реализуем экспоненциальный случай. Разбиваем область под убывающей плотностью на слои. Большая часть каждого слоя - прямоугольная область, в которой кандидат принимается сразу. У внешнего края проверяем попадание под кривую. Для хвоста за порогом $R$ используем $R+E$, где $E\sim\operatorname{Exp}(1)$: у экспоненциального закона остаточное время ожидания имеет тот же закон.

**Где используется.** Для многократной генерации времени ожидания в моделях очередей или событий. Ziggurat для нормального и экспоненциального законов применяется и в генераторах NumPy.

**Сильная сторона.** Большинство предложений обрабатывается без логарифма и без вычисления плотности. **Слабая сторона.** Нужны таблица границ, обработка края и хвоста. Здесь используем учебную реализацию на Python и не сравниваем её скорость с библиотечной.

**Источники:** Дж. Марсалья и В. Цанг, [*The Ziggurat Method for Generating Random Variables* и исходный алгоритм на C](https://doi.org/10.18637/jss.v005.i08), 2000; [документация NumPy](https://numpy.org/doc/stable/reference/random/new-or-different.html).

In [ ]:
# Ziggurat: экспоненциальное время ожидания
def exponential_ziggurat(rng, size):
    # Граница хвоста для 256 слоёв из приложения к статье.
    r = 7.697117470131487
    layers = 256
    area = (r + 1) * math.exp(-r)
    bounds = [r]
    for _ in range(layers - 2):
        x = bounds[-1]
        bounds.append(-math.log(math.exp(-x) + area/x))
    bounds.append(0.0)
    heights = np.exp(-np.array(bounds))
    # Проверяем площади слоёв. Нулевой слой включает хвост.
    np.testing.assert_allclose(np.array(bounds[:-1])*np.diff(heights), area,
                               rtol=1e-7, atol=0)

    result = np.empty(size)
    for i in range(size):
        while True:
            layer = rng.integers(layers)
            if layer == 0:
                x = (r+1)*rng.random()
                result[i] = x if x < r else r-math.log1p(-rng.random())
                break
            x = bounds[layer-1]*rng.random()
            if x < bounds[layer]:
                result[i] = x
                break
            height = heights[layer-1] + rng.random()*(heights[layer]-heights[layer-1])
            if height <= math.exp(-x):
                result[i] = x
                break
    return result


exponential = exponential_ziggurat(np.random.default_rng(108), N)
print("Пример интервалов между событиями:", np.round(exponential[:6], 3))

Пример интервалов между событиями: [4.867 0.077 1.244 0.169 4.151 0.776]


## Результаты контрольных запусков

В каждом примере получаем 20 000 значений. Для дискретной влажности категории закодированы числами 0, 1, 2 в порядке возрастания. Поэтому сравниваем накопленные вероятности по этим категориям и отдельно долю среднего диапазона. Для остальных методов сравниваем среднее, дисперсию и накопленные вероятности с известным законом. Для непрерывных законов смотрим разрыв на сетке: это учебная проверка, не формальный тест согласия.

Среднее и дисперсия для первых четырёх методов относятся к **кодам категорий**, а не к влажности в процентах. Скорость не сравниваем. Если нужны большие серии из неизменных дискретных частот, имеет смысл рассмотреть псевдонимы. Если хотим варьировать доли стран, используем суперпозицию. Если есть теоретический закон и задача выходит за рамки влажности, выбираем метод под этот закон.

In [ ]:
# Таблица результатов для восьми алгоритмов
rows = []


def report(name, sample, grid, theoretical_cdf, target_mean, target_var):
    assert len(sample) == N and np.isfinite(sample).all()
    empirical_cdf = np.searchsorted(np.sort(sample), grid, side="right") / N
    gap = np.max(np.abs(empirical_cdf - theoretical_cdf))
    assert gap < .02, (name, gap)
    rows.append([name, f"{sample.mean():.3f} / {target_mean:.3f}",
                 f"{sample.var():.3f} / {target_var:.3f}", f"{100*gap:.2f}"])


for name, sample, p in [("Обратное преобразование", s1_inverse, p_all),
                        ("Псевдонимы", s1_alias, p_all),
                        ("Отбор с отклонением", s1_reject, p_all),
                        ("Суперпозиция", s1_mix, p_mix)]:
    assert np.isin(sample, levels).all()
    mean = levels @ p
    report(name, sample, levels, p.cumsum(), mean, (levels - mean)**2 @ p)
    print(f"{name}: 10–12% - {100*np.mean(sample == 1):.2f}% / {100*p[1]:.2f}%")

x = np.linspace(-5, 5, 401)
report("Бокс–Мюллер", normal, x,
       np.array([(1 + math.erf(t/math.sqrt(2)))/2 for t in x]), 0, 1)
x = np.linspace(0, 15, 601)
assert (gamma >= 0).all()
report("Марсалья–Цанг", gamma, x, 1-(1+x)*np.exp(-x), 2, 2)
k = np.arange(31)
assert (poisson >= 0).all()
report("Кнут", poisson, k, np.cumsum([math.exp(-4)*4**int(i)/math.factorial(int(i)) for i in k]), 4, 4)
assert (exponential >= 0).all()
report("Ziggurat", exponential, x, -np.expm1(-x), 1, 1)
assert len(rows) == 8

headers = ["Метод", "Среднее: запуск / теория", "Дисперсия: запуск / теория", "Макс. разрыв CDF, п. п."]
from IPython.display import Markdown, display

display(Markdown("\n".join("| " + " | ".join(row) + " |"
                           for row in [headers, ["---"]*len(headers), *rows])))

Обратное преобразование: 10–12% — 79.59% / 79.77%
Псевдонимы: 10–12% — 79.99% / 79.77%
Отбор с отклонением: 10–12% — 79.64% / 79.77%
Суперпозиция: 10–12% — 71.45% / 71.95%


| Метод | Среднее: запуск / теория | Дисперсия: запуск / теория | Макс. разрыв CDF, п. п. |
| --- | --- | --- | --- |
| Обратное преобразование | 1.027 / 1.028 | 0.203 / 0.201 | 0.17 |
| Псевдонимы | 1.028 / 1.028 | 0.199 / 0.201 | 0.11 |
| Отбор с отклонением | 1.031 / 1.028 | 0.203 / 0.201 | 0.22 |
| Суперпозиция | 1.243 / 1.238 | 0.227 / 0.224 | 0.48 |
| Бокс–Мюллер | 0.008 / 0.000 | 1.005 / 1.000 | 0.55 |
| Марсалья–Цанг | 2.004 / 2.000 | 2.004 / 2.000 | 0.29 |
| Кнут | 4.000 / 4.000 | 4.065 / 4.000 | 0.31 |
| Ziggurat | 0.992 / 1.000 | 0.969 / 1.000 | 0.44 |